## AWS External Functions in Snowflake
**Snowflake Advanced 3-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Create and test an AWS Lambda function
2. Connect an IAM-protected API Gateway REST API
3. Create a Snowflake API integration and establish AWS trust
4. Call remote Python logic from SQL, including a batch and NULL input
5. Verify against the course orders and clean up both platforms

**Run one cell at a time. Do not use Run All.** AWS console actions between cells are required. Execute Python cells in a Snowflake-hosted notebook with `snowflake-snowpark-python` available. The notebook generates policies and SQL; it does not use AWS credentials or deploy AWS resources automatically.

The request path is **SQL → external function / API integration → API Gateway → Lambda → SQL result**. This is an external function, not a Python UDF using an external access integration. The risk rule is deliberately simple; the lesson is the cross-platform connection, not predictive risk analysis.

**Prerequisites:** AWS permissions for Lambda, API Gateway REST APIs, IAM roles/policies and CloudWatch logs; Snowflake ACCOUNTADMIN for integration setup; existing course role `ADV_LAB`, warehouse `ADV_WH`, and database `ADV_ORDERS`. Use dedicated training resources. AWS services and Snowflake compute can incur charges.


**Workflow:** notebook preparation → AWS resources and empty invocation role → notebook integration and both policy documents → AWS policies → notebook function and checks → final AWS log review and cleanup. Keep notebook and AWS tabs side by side for copying prepared text. No separate SQL worksheet, local Python environment or AWS CLI is required.


### Reset a previous run (skip on first use)
Use this section only if you already created this demo. It removes the previous demo so the same final names can be used from start to finish. Do not delete course database objects beyond the specific function and integration listed below.

**Snowflake:** execute the following SQL cell to remove the previous demo function and integration. Skip this cell on a first run.


In [ ]:
%%sql -r external_reset
USE ROLE ACCOUNTADMIN;
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.DEMO_ORDER_RISK(FLOAT);
DROP INTEGRATION IF EXISTS ORDER_RISK_API_INTEGRATION;

**AWS console**, in the region used for the previous run (use the region selected for your demo):
1. API Gateway: find `snowflake-order-risk-api`, verify its identity, and delete the REST API and its stages. Confirm the API ID belongs to this demo before deleting it.
2. Lambda: open `snowflake-order-risk-demo`. Record its execution role under **Configuration → Permissions**, then use **Actions → Delete** to remove the function.
3. IAM: delete `SnowflakeOrderRiskInvokeRole` and its inline invocation policy. Delete the recorded Lambda execution role only if it was created solely for this demo and is not shared. Do not delete AWS-managed policies; remove a dedicated customer-managed logging policy only if it is unused.
4. CloudWatch Logs: delete `/aws/lambda/snowflake-order-risk-demo` if the previous logs are no longer needed.

**CHECKPOINT:** the demo API, Lambda, dedicated IAM roles, Snowflake function and API integration have been removed. Now start Step 1 using the final names. The newly created API URL and Snowflake-generated external ID must be obtained again during this run.

For cleanup after a completed run, use the final cleanup section, which uses that run's configured names.


### STEP 1 | Prepare names, code and test payloads in the notebook
Run the setup cells, then use the static Lambda code and test JSON blocks below. Keep this section beside the AWS console; no notebook execution is needed during the first AWS visit.

Use the same final names on every run. If resources already exist, complete the reset section first. Enter your AWS account ID as a string to preserve leading zeros. Python generates the JSON and parameterized SQL; SQL executes directly through this notebook's Snowflake session.


In [ ]:
%%sql -r extfn_context
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ADV_WH;
USE DATABASE ADV_ORDERS;
USE SCHEMA OPS;
ALTER SESSION SET USE_CACHED_RESULT = FALSE;
SELECT CURRENT_ROLE(), CURRENT_REGION(), CURRENT_WAREHOUSE();

In [ ]:
import json
import re
from urllib.parse import urlparse
from snowflake.snowpark.context import get_active_session

session = get_active_session()

aws_account_id = "024855656562"  # Replace for another AWS account; preserve leading zero.
aws_region = "us-east-2"
lambda_name = "snowflake-order-risk-demo"
api_name = "snowflake-order-risk-api"
aws_role_name = "SnowflakeOrderRiskInvokeRole"
inline_policy_name = "InvokeOrderRiskDemo"
stage_name = "demo"
integration_name = "ORDER_RISK_API_INTEGRATION"
function_name = "ADV_ORDERS.OPS.DEMO_ORDER_RISK"

assert re.fullmatch(r"\d{12}", aws_account_id)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*", integration_name)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*(\.[A-Z][A-Z0-9_]*){2}", function_name)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

print("Lambda:", lambda_name)
print("API:", api_name)
print("IAM role:", aws_role_name)
print("Snowflake function:", function_name)


**Lambda code — copy into `lambda_function.py`:**

```python
import json

def lambda_handler(event, context):
    rows = json.loads(event["body"])["data"]
    results = []

    for row_number, amount in rows:
        if amount is None:
            risk = "UNKNOWN"
        elif amount > 100:
            risk = "REVIEW"
        else:
            risk = "OK"
        results.append([row_number, risk])

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"data": results})
    }
```

**Lambda test event:**

```json
{
  "body": "{\"data\": [[0, 120], [1, 45], [2, null]]}"
}
```

**API Gateway request body:**

```json
{
  "data": [
    [
      0,
      120
    ],
    [
      1,
      45
    ],
    [
      2,
      null
    ]
  ]
}
```

### STEP 2 | Create the Lambda function in AWS
In the AWS console, select the configured region and open **Lambda**.

1. On the welcome page, click **Create an Event Function**. The “How it works” editor is only an introductory example.
2. Choose **Author from scratch**.
3. Enter the Lambda name printed above; select **Python 3.12**.
4. Leave **Durable execution** and **EC2 capacity provider** off. Leave **Additional settings** at their defaults.
5. The Permissions text says Lambda automatically creates an execution role with CloudWatch Logs permissions. Keep that default.
6. Click **Create Event Function**; dismiss the **Getting started** popup.
7. Open **Code → lambda_function.py**. Replace the starter code with the Lambda code prepared in Step 1, then click **Deploy** in the editor's left panel.

This execution role lets Lambda write logs. A separate IAM role will later authorize Snowflake to invoke API Gateway.


### STEP 3 | Test Lambda before connecting the gateway
1. Under **TEST EVENTS [NONE SELECTED]**, click **Create new test event**.
2. Name it `snowflake_batch_test`; choose **Private**.
3. Replace Event JSON with the **LAMBDA TEST EVENT** prepared in Step 1.
4. Save the event. If the save action is not visible, scroll **inside the right-hand test-event form**, outside the JSON editor; the form has its own scrollbar.
5. Click the blue **Test** button below Deploy.

**CHECKPOINT:** statusCode `200`; response body contains `[[0,"REVIEW"],[1,"OK"],[2,"UNKNOWN"]]` under `data`. The body is a JSON-encoded string, so escaped quotes are normal.

Snowflake sends batches even though SQL expresses one function result per row. The first item in each input/output row is a **batch row number**, not an order ID. Return one result per input, preserving those numbers and order. Three inputs simply exercise the three branches; batch size is not fixed at three. Keep the handler free of side effects because remote requests can be retried.


### STEP 4 | Create the API Gateway REST API
1. Search for **API Gateway** in the AWS console. Keep the same region as Lambda.
2. Choose **Create API → REST API → Build** (not HTTP API or REST API Private).
3. Select **New API**, enter the configured API name, and leave Description optional.
4. Set **Endpoint type = Regional** and **IP address type = IPv4**.
5. Select **SecurityPolicy_TLS13_1_2_2021_06**, supporting TLS 1.2 and TLS 1.3.
6. Click **Create API**.

The TLS policy controls transport encryption; IAM authorization is configured separately below.


### STEP 5 | Add the root POST method
On the Resources page, keep the root `/` selected and click **Create method**.

| Setting | Value |
|---|---|
| Method type | POST |
| Integration type | Lambda function |
| Lambda proxy integration | Enabled |
| Response transfer mode | Buffered |
| Lambda region | The configured region |
| Lambda function | The function created in Step 2 |
| Integration timeout | 29000 ms (default) |

Expand **Method request settings**:
- Authorization: **AWS IAM**
- Request validator: **None**
- API key required: **unchecked**
- Operation name: leave blank (the displayed `GetPets` text is a placeholder)

Leave URL query strings, HTTP headers and request-body configuration unchanged. Click **Create method**. The console grants API Gateway permission to invoke Lambda through Lambda's resource-based policy.

**CHECKPOINT:** the POST Method execution page shows **Authorization = AWS_IAM**. Lambda proxy integration passes the HTTP body to our handler as `event["body"]`.


### STEP 6 | Test API Gateway, then deploy
1. On the POST method page, select **Test**, to the right of Method response.
2. Paste the **API Gateway request body** prepared in Step 1 into **Request body**; leave other optional fields empty.
3. Click **Test**.

Unlike the Lambda test, this is the raw HTTP body: do not add the outer `body` wrapper. API Gateway supplies that wrapper.

**CHECKPOINT:** HTTP `200` and the same REVIEW / OK / UNKNOWN results. This proves gateway-to-Lambda connectivity, not the Snowflake IAM trust relationship.

4. Click **Deploy API** at the upper right.
5. Choose **New stage**, name it `demo`, and click **Deploy**.
6. Copy the stage's **Invoke URL** for Step 8. Stay in AWS and create the IAM role in Step 7 before returning to the notebook. We use the root resource, so no additional resource path is appended.

An API Gateway stage is a deployment environment; it is unrelated to a Snowflake file stage. Changes to methods or API policies may require redeployment to this stage.


### STEP 7 | Create the invocation role before leaving AWS
1. Open **IAM → Roles → Create role**.
2. Choose **AWS account → This account** as the temporary trusted entity.
3. Click **Next**; select no permission policies and click **Next** again.
4. Enter `SnowflakeOrderRiskInvokeRole` (or the configured role name) and create it.
5. Record its ARN from the summary page.

Do not add the invocation policy yet. We will generate both final policies together after Snowflake creates its integration identity. The new role has no API invocation permissions at this point.

**Return to the notebook now**, with the deployed Invoke URL and the role ARN. This ends the first AWS visit.


### STEP 8 | Record the endpoint and verify the role ARN
Paste the Invoke URL from this run below. A recreated API receives a new URL. The cell constructs the exact POST permission and role ARN; compare the printed role ARN with the AWS role summary. If they differ, correct the configured account ID or role name before continuing.


In [ ]:
invoke_url = "PASTE_INVOKE_URL_HERE"
parsed = urlparse(invoke_url)
assert parsed.scheme == "https", "Paste the deployed API Invoke URL."
assert parsed.hostname and parsed.hostname.endswith(f".execute-api.{aws_region}.amazonaws.com")
assert parsed.path.rstrip("/") == f"/{stage_name}"
assert not parsed.query and not parsed.fragment
invoke_url = invoke_url.rstrip("/")
api_id = parsed.hostname.split(".")[0]
aws_role_arn = f"arn:aws:iam::{aws_account_id}:role/{aws_role_name}"
invoke_arn = f"arn:aws:execute-api:{aws_region}:{aws_account_id}:{api_id}/{stage_name}/POST/"
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": "execute-api:Invoke", "Resource": invoke_arn}]
}
print("IAM role:", aws_role_name)
print("Policy name:", inline_policy_name)
print(json.dumps(invoke_policy, indent=2))

print("Verify against the AWS role summary:", aws_role_arn)


### STEP 9 | Create the Snowflake API integration
The integration references the IAM role and limits the permitted URL prefix. Run this cell once per fresh run. It intentionally uses **CREATE**, not CREATE OR REPLACE: replacing an integration can change the external ID and require updating AWS trust again.

If the object already exists, use the following DESCRIBE cell to inspect it, or finish cleanup before starting a fresh run.


In [ ]:
%%sql -r extfn_admin
USE ROLE ACCOUNTADMIN;

In [ ]:
integration_ddl = f"""CREATE API INTEGRATION {integration_name}
    API_PROVIDER = aws_api_gateway
    API_AWS_ROLE_ARN = {sql_literal(aws_role_arn)}
    API_ALLOWED_PREFIXES = ({sql_literal(invoke_url)})
    ENABLED = TRUE"""
print(integration_ddl)
session.sql(integration_ddl).collect()
print("API integration created.")


In [ ]:
properties = {
    row["property"]: row["property_value"]
    for row in session.sql(f"DESCRIBE INTEGRATION {integration_name}").collect()
}
sf_iam_user_arn = properties["API_AWS_IAM_USER_ARN"]
external_id = properties["API_AWS_EXTERNAL_ID"]
print("API_AWS_IAM_USER_ARN:", sf_iam_user_arn)
print("API_AWS_EXTERNAL_ID:", external_id)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": sf_iam_user_arn},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"sts:ExternalId": external_id}}
    }]
}
print("\nReplace the IAM role trust policy with:")
print(json.dumps(trust_policy, indent=2))

print("\nINLINE INVOCATION POLICY — name:", inline_policy_name)
print(json.dumps(invoke_policy, indent=2))


### STEP 10 | Apply both policies in one AWS visit
Open **IAM → Roles → your invocation role**.

1. Under **Permissions → Add permissions → Create inline policy → JSON**, paste the **INLINE INVOCATION POLICY** printed above.
2. Click **Next**, name it `InvokeOrderRiskDemo` (or the configured policy name), and create it.
3. Open **Trust relationships → Edit trust policy**.
4. Replace the entire temporary trust policy with the **trust policy** printed above, and click **Update policy**.

The permissions policy grants invocation of only this API/stage/POST/root path. The trust policy allows the exact Snowflake principal to assume the role with the generated external ID. Do not modify Lambda's execution role.

**CHECKPOINT:** both policies are saved. Return to the notebook for Steps 11–14. No further AWS visit is required until final log inspection and cleanup. IAM changes may take a short time to propagate.


### STEP 11 | Create the external function
Use the existing course role for function ownership and execution. The definition is stored in Snowflake; its Python implementation runs in AWS Lambda. `CALLED ON NULL INPUT` makes the NULL branch explicit. `VOLATILE` reflects that the remote implementation can change.


In [ ]:
%%sql -r extfn_grants
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON INTEGRATION ORDER_RISK_API_INTEGRATION TO ROLE ADV_LAB;
USE ROLE ADV_LAB;
USE SECONDARY ROLES NONE;
USE WAREHOUSE ADV_WH;

In [ ]:
function_ddl = f"""CREATE OR REPLACE EXTERNAL FUNCTION {function_name}(AMOUNT FLOAT)
RETURNS VARCHAR
CALLED ON NULL INPUT
VOLATILE
API_INTEGRATION = {integration_name}
AS {sql_literal(invoke_url)}"""
print(function_ddl)
session.sql(function_ddl).collect()
print("External function created.")


### STEP 12 | Verify the complete connection
The next query tests above-threshold, below-threshold, boundary, and NULL values. The client row IDs in VALUES make the output easy to check; they are not the internal batch row numbers sent over HTTP.

**CHECKPOINT:** 120 → REVIEW, 45 → OK, 100 → OK, NULL → UNKNOWN. A successful Lambda-console or API-console test alone is not sufficient; this SQL test verifies Snowflake authentication and the full request path.


In [ ]:
%%sql -r extfn_022
SELECT COLUMN1 AS TEST_ID, COLUMN2 AS AMOUNT,
    ADV_ORDERS.OPS.DEMO_ORDER_RISK(COLUMN2::FLOAT) AS RISK_LABEL,
    COLUMN3 AS EXPECTED_LABEL,
    IFF(RISK_LABEL = EXPECTED_LABEL, 'PASS', 'FAIL') AS RESULT
FROM VALUES (1, 120, 'REVIEW'), (2, 45, 'OK'),
            (3, 100, 'OK'), (4, NULL, 'UNKNOWN')
ORDER BY TEST_ID;

### STEP 13 | Apply the function to course orders
This is a read-only demonstration. No pipeline table or task is modified. The query compares the remote output with our deliberately simple local SQL rule, so it does not depend on fixed order amounts.


In [ ]:
%%sql -r extfn_024
SELECT ORDER_ID, AMOUNT,
    ADV_ORDERS.OPS.DEMO_ORDER_RISK(AMOUNT::FLOAT) AS RISK_LABEL,
    CASE WHEN AMOUNT IS NULL THEN 'UNKNOWN'
         WHEN AMOUNT > 100 THEN 'REVIEW' ELSE 'OK' END AS EXPECTED_LABEL,
    IFF(RISK_LABEL = EXPECTED_LABEL, 'PASS', 'FAIL') AS RESULT
FROM ADV_ORDERS.SILVER.ORDERS_CURRENT
ORDER BY ORDER_ID;
-- CHECKPOINT: rows are returned and every RESULT is PASS.

### STEP 14 | Explain the execution boundary
The SQL checkpoints above are the end-to-end checkpoint. Stay in the notebook here; inspect CloudWatch logs during the final AWS cleanup visit below. The default handler does not print input data, and Snowflake can batch rows, so invocation counts need not equal row counts.

Discussion:
- Snowflake manages SQL execution and authenticates through the API integration.
- API Gateway checks IAM authorization and forwards the request.
- Lambda runs the Python rule and returns batched results.
- Network calls introduce latency and failure modes. Use this for remote capabilities/services, not as a preferred replacement for a simple SQL CASE.

**Troubleshooting:**
- Access denied while creating the integration: check the Snowflake role.
- Cannot assume role: compare the IAM trust principal and external ID with DESCRIBE INTEGRATION; allow IAM propagation time.
- HTTP 403: check API authorization, invocation role permissions, API/stage/method ARN, and deployed URL.
- Missing Authentication Token: check stage/path and method; a browser issues GET but this API exposes POST.
- HTTP 500/502: check Lambda logs, proxy integration, and the response's string-valued `body`.
- KeyError `body` in Lambda: distinguish the wrapped Lambda test event from the raw API Gateway request body.
- After changing API methods/policies, redeploy to `demo` before testing again.


### STEP 15 | Clean up Snowflake objects in the notebook
Run cleanup **after** verifying all checkpoints. The following SQL cell removes only the lab external function and API integration. Run it when finished; it does not remove course tables, roles, or the warehouse.

Dropping Snowflake objects does **not** delete AWS resources. Complete the AWS console cleanup below as well.


In [ ]:
%%sql -r extfn_027
-- Execute only after completing all checks.
USE ROLE ACCOUNTADMIN;
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.DEMO_ORDER_RISK(FLOAT);
DROP INTEGRATION IF EXISTS ORDER_RISK_API_INTEGRATION;

In [ ]:
print("AWS cleanup targets:")
print("Region:", aws_region)
print("API Gateway REST API:", api_name, "| API ID:", api_id)
print("Lambda:", lambda_name)
print("Invocation IAM role:", aws_role_name)
print("Inline policy:", inline_policy_name)
print("CloudWatch log group:", "/aws/lambda/" + lambda_name)


### STEP 16 | Inspect AWS logs and clean up in one visit
Before deletion, open the configured Lambda → **Monitor → View CloudWatch logs** to inspect invocation entries. Then complete all AWS cleanup in this visit.

1. **API Gateway:** select the REST API by the printed name **and API ID**, then delete that API using its actions menu and confirm. This removes its deployment stages as well.
2. **Lambda:** before deleting, record the auto-created execution role under **Configuration → Permissions**. Delete the configured Lambda through **Actions → Delete**.
3. **IAM:** delete the dedicated invocation role and its inline policy. Delete the auto-created Lambda execution role only after confirming it belongs solely to this demonstration. Do not delete AWS-managed policies or a shared execution role. Remove a dedicated customer-managed logging policy only if it is unused and was created for this Lambda.
4. **CloudWatch → Logs → Log groups:** delete `/aws/lambda/<configured Lambda name>` if you do not need its logs. Lambda deletion does not automatically remove retained logs.
5. Confirm the API, Lambda and dedicated IAM roles are gone. If no other course work is running, use the course shutdown script to suspend course compute and background workloads; close the notebook session separately.

**To rehearse from scratch:** complete both Snowflake and AWS cleanup, restart the notebook kernel/session, start at Step 1. The new API gets a new ID and URL: update Step 8. Recreate the integration and regenerate the trust policy; do not reuse the old external ID. Execute each cell in order, pausing for the AWS console steps. Keep cleanup until the end.

### References
- [Snowflake: create an external function using the AWS console](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui)
- [Snowflake: Lambda setup and batch data format](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui-remote-service)
- [Snowflake: API integration](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-common-api-integration)
- [AWS: API Gateway security policies](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-security-policies-list.html)
